In [0]:
%pylab inline

In [1]:
import dataiku
import pyspark
import pandas as pd
import dataiku.spark as dkuspark
from pyspark.sql import SQLContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

spark = SparkSession.builder.appName("").getOrCreate()
sc = spark.sparkContext
sqlContext = SQLContext(sc)

/data/dataiku/dss_data/code-envs/python/QB_DE_OmniChannel/lib/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)
/opt/dataiku-dss-13.2.0/spark-standalone-home/python/pyspark/context.py:317: FutureWarning: Python 3.7 support is deprecated in Spark 3.4.
  warnings.warn("Python 3.7 support is deprecated in Spark 3.4.", FutureWarning)
/opt/dataiku-dss-13.2.0/spark-standalone-home/python/pyspark/sql/context.py:114: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


# 1. Load Data

Reads the six Dataiku datasets into Spark DataFrames:
- mx: medical claims
- rx: pharmacy claims
- ddd: DDD sales
- opal: OPAL hierarchy
- cm: ChargeMaster
- dmn: dmn_cust, the NPI → J&J ID bridge
- Prints the row and column count of each table.

In [2]:
def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))


mx   = load("MX_RYB_CLAIMS_copy")        # medical claims (infusions)
rx   = load("RYB_LINK_CLAIMS_copy")      # pharmacy claims
ddd  = load("RYB_DDD_Sales_copy")        # DDD sales, weekly, jnj_id = L1 outlet
opal = load("OPAL_Affiliation_Levels")   # HCP -> L1..L5 parents (wide)
cm   = load("ChargeMaster_WeeklyReporting_20261012_v2_copy")
dmn  = load("dmn_cust_copy") # DMN_CUST: customer master, the NPI -> J&J ID bridge (npi_id -> jnj_id).

for name, df in [("mx", mx), ("rx", rx), ("ddd", ddd), ("opal", opal), ("chargemaster", cm), ("dmn_cust", dmn)]:
    if df is None:
        continue
    print(f"{name:13s} rows={df.count():>11,}  cols={len(df.columns)}")

/opt/dataiku-dss-13.2.0/spark-standalone-home/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


mx            rows=        983  cols=28
rx            rows=         15  cols=33
ddd           rows=      3,419  cols=5
opal          rows=  4,146,177  cols=8
chargemaster  rows=        571  cols=33
dmn_cust      rows= 22,310,923  cols=70


# 2. Profile each table

For each table it shows:
- the number of rows;
- the date range (earliest and latest date);
- the number of distinct values of each ID column;
- how many rows have that ID empty.

- Patients in the claims are counted as distinct dv1 + dv2.
- Look at: the date ranges (what period we really have), and the empty-ID counts (they limit how much can be linked).

In [4]:
from pyspark.sql import functions as F
def blank(c):
    col = F.trim(F.col(f"`{c}`").cast("string"))
    return col.isNull() | (col == "")

def profile(name, df, date_cols, id_cols):
    aggs = [F.count("*").alias("rows")]
    for c in date_cols:
        aggs += [F.min(F.to_date(F.col(f"`{c}`"))).cast("string").alias(f"min {c}"),
                 F.max(F.to_date(F.col(f"`{c}`"))).cast("string").alias(f"max {c}")]
    for c in id_cols:
        aggs += [F.countDistinct(F.col(f"`{c}`")).alias(f"distinct {c}"),
                 F.sum(blank(c).cast("int")).alias(f"empty {c}")]
    print(f"\n=== {name} ===")
    df.agg(*aggs).show(vertical=True, truncate=False)

profile("MX claims", mx.withColumn("patient", F.concat_ws("|", "dv1", "dv2")), ["service_date"],
        ["patient", "mx_claim_id", "ndc", "procedure_code", "rendering_npi", "referring_npi", "billing_npi", "facility_npi"])
profile("RX claims", rx.withColumn("patient", F.concat_ws("|", "dv1", "dv2")), ["fill_date", "written_date"],
        ["patient", "claim_id", "ndc", "prescriber_npi", "pharmacy_npi"])
profile("DDD", ddd, ["caln_dt"], ["jnj_id", "prod_lst_id", "prod_nm"])
profile("OPAL", opal, [], ["0", "1", "2", "3", "4", "5", "L2_L3"])
#profile("dmn_cust", dmn, ["efftv_dt", "end_dt"], ["npi_id", "jnj_id"])


=== MX claims ===
-RECORD 0-----------------------------
 rows                    | 983        
 min service_date        | 2026-08-03 
 max service_date        | 2026-09-01 
 distinct patient        | 599        
 empty patient           | 0          
 distinct mx_claim_id    | 983        
 empty mx_claim_id       | 0          
 distinct ndc            | 6          
 empty ndc               | 0          
 distinct procedure_code | 4          
 empty procedure_code    | 4          
 distinct rendering_npi  | 505        
 empty rendering_npi     | 6          
 distinct referring_npi  | 74         
 empty referring_npi     | 850        
 distinct billing_npi    | 208        
 empty billing_npi       | 2          
 distinct facility_npi   | 0          
 empty facility_npi      | 983        


=== RX claims ===
-RECORD 0-----------------------------
 rows                    | 15         
 min fill_date           | 2026-07-06 
 max fill_date           | 2026-07-30 
 min written_date        

# 3. Keep Rybrevant only

Claims: the product comes from the NDC reference in RYB_HN_Data_Sources.csv.
- Rybrevant NDCs: 57894050100, 57894050101.
- Rybrevant Faspro NDCs: 57894051001, 57894051401, 57894051501, 57894052201.
- If a claim's NDC is not in that list, the claim's own product_name is used when it says RYBREVANT.
- Anything else (Lazcluze NDCs, other) is dropped.
- DDD: keeps rows whose prod_nm contains RYBREVANT (Rybrevant and Rybrevant Faspro); Lazcluze rows are dropped.
- Shows what was kept and what was dropped, so you can see every NDC / product name that exists.

In [5]:
RYB_NDC = {   # from RYB_HN_Data_Sources.csv (NDC Reference)
    "57894050100": "RYBREVANT", "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT FASPRO", "57894051401": "RYBREVANT FASPRO",
    "57894051501": "RYBREVANT FASPRO", "57894052201": "RYBREVANT FASPRO",
}
ndc_map = F.create_map(*[F.lit(x) for kv in RYB_NDC.items() for x in kv])

def ryb_product(ndc_col, name_col):
    name = F.upper(F.trim(F.col(name_col)))
    from_name = F.when(name.contains("RYBREVANT"), name)        # the claim's own product name
    return F.coalesce(ndc_map[F.trim(F.col(ndc_col))], from_name)

mx_p = mx.withColumn("product", ryb_product("ndc", "product_name"))
rx_p = rx.withColumn("product", ryb_product("ndc", "product_name"))

print("MX claims: kept vs dropped by NDC / procedure code / product name")
(mx_p.groupBy(F.when(F.col("product").isNotNull(), "kept").otherwise("dropped").alias("status"),
              "ndc", "procedure_code", "product_name", "product").count().orderBy("status", F.desc("count")).show(50, False))
print("RX claims: kept vs dropped by NDC / product name")
(rx_p.groupBy(F.when(F.col("product").isNotNull(), "kept").otherwise("dropped").alias("status"),
              "ndc", "product_name", "product").count().orderBy("status", F.desc("count")).show(50, False))

ddd_p = ddd.withColumn("product", F.upper(F.trim("prod_nm")))
print("DDD: kept vs dropped by product")
(ddd_p.groupBy(F.when(F.col("product").contains("RYBREVANT"), "kept").otherwise("dropped").alias("status"),
               "prod_lst_id", "prod_nm")
      .agg(F.count("*").alias("rows"), F.round(F.sum("equiv_uns"), 2).alias("equiv_uns"))
      .orderBy("status", "prod_nm").show(50, False))

mx_r  = mx_p.filter(F.col("product").isNotNull())
rx_r  = rx_p.filter(F.col("product").isNotNull())
ddd_r = ddd_p.filter(F.col("product").contains("RYBREVANT"))
print(f"kept rows -> MX {mx_r.count():,} of {mx.count():,} | RX {rx_r.count():,} of {rx.count():,} | DDD {ddd_r.count():,} of {ddd.count():,}")

MX claims: kept vs dropped by NDC / procedure code / product name
+------+-----------+--------------+----------------+----------------+-----+
|status|ndc        |procedure_code|product_name    |product         |count|
+------+-----------+--------------+----------------+----------------+-----+
|kept  |57894050101|J9061         |RYBREVANT       |RYBREVANT       |326  |
|kept  |57894051001|J9062         |RYBREVANT FASPRO|RYBREVANT FASPRO|306  |
|kept  |57894051501|J9062         |RYBREVANT FASPRO|RYBREVANT FASPRO|119  |
|kept  |57894051401|J9062         |RYBREVANT FASPRO|RYBREVANT FASPRO|77   |
|kept  |57894052201|J9062         |null            |RYBREVANT FASPRO|73   |
|kept  |57894050100|J9061         |RYBREVANT       |RYBREVANT       |71   |
|kept  |57894050101|null          |RYBREVANT       |RYBREVANT       |4    |
|kept  |57894051001|C9399         |RYBREVANT FASPRO|RYBREVANT FASPRO|2    |
|kept  |57894051001|J9999         |RYBREVANT FASPRO|RYBREVANT FASPRO|2    |
|kept  |57894051401|J9

# Remove pharmacy claims of patients who are already in the medical claims
Rule: a patient found in the medical claims is taken from the medical claims only. Their pharmacy claims are dropped, so no patient is counted twice.

- Patients who appear only in the pharmacy claims are kept from the pharmacy claims (doctor = prescriber_npi).
- A patient is dv1 + dv2, the same tokens in both tables.

Prints:
- the RX patients and rows before the rule;
- how many of them are also in MX (dropped);
- how many are RX-only (kept).
- From here on, rx_r holds only the RX-only patients.

In [6]:
mx_patients = mx_r.select(F.concat_ws("|", "dv1", "dv2").alias("patient_id")).distinct()
rx_all = rx_r.withColumn("patient_id", F.concat_ws("|", "dv1", "dv2"))
rx_in_mx = rx_all.join(mx_patients, "patient_id", "left_semi")
rx_r = rx_all.join(mx_patients, "patient_id", "left_anti").drop("patient_id").cache()

print(f"RX before rule : {rx_all.select('patient_id').distinct().count():,} patients, {rx_all.count():,} rows")
print(f"  also in MX   : {rx_in_mx.select('patient_id').distinct().count():,} patients, {rx_in_mx.count():,} rows  -> dropped")
print(f"  RX only      : {rx_r.select(F.concat_ws('|', 'dv1', 'dv2')).distinct().count():,} patients, {rx_r.count():,} rows  -> kept")
     

RX before rule : 13 patients, 15 rows
  also in MX   : 2 patients, 2 rows  -> dropped
  RX only      : 11 patients, 13 rows  -> kept


# put every NPI on the claims into one long list

- A claim carries several NPIs. The doctor on a medical claim is rendering_npi. All NPIs are still kept, each with its role, so the other roles stay visible for checks:
- medical claims: rendering_npi, referring_npi, billing_npi, facility_npi;
- pharmacy claims: prescriber_npi, pharmacy_npi.
- Result claim_npis: one row per claim × NPI role, with patient, date and product.
- Later sections report coverage separately for each role

In [7]:
def npi_rows(df, source, claim_id, date_col, roles):
    parts = []
    for role in roles:
        parts.append(df.select(F.lit(source).alias("source"), F.col(claim_id).alias("claim_id"),
                               F.concat_ws("|", "dv1", "dv2").alias("patient_id"),
                               F.to_date(date_col).alias("service_dt"), "product",
                               F.lit(role).alias("npi_role"), F.trim(F.col(role).cast("string")).alias("npi")))
    out = parts[0]
    for p in parts[1:]:
        out = out.unionByName(p)
    return out.filter(F.col("npi").isNotNull() & (F.col("npi") != ""))

claim_npis = (npi_rows(mx_r, "MX", "mx_claim_id", "service_date",
                       ["rendering_npi", "referring_npi", "billing_npi", "facility_npi"])
              .unionByName(npi_rows(rx_r, "RX", "claim_id", "fill_date", ["prescriber_npi", "pharmacy_npi"]))
              .cache())
(claim_npis.groupBy("source", "npi_role")
           .agg(F.count("*").alias("claim_rows"), F.countDistinct("npi").alias("distinct_npis"),
                F.countDistinct("patient_id").alias("patients"))
           .orderBy("source", "npi_role").show(truncate=False))

+------+--------------+----------+-------------+--------+
|source|npi_role      |claim_rows|distinct_npis|patients|
+------+--------------+----------+-------------+--------+
|MX    |billing_npi   |981       |208          |598     |
|MX    |referring_npi |133       |74           |88      |
|MX    |rendering_npi |977       |505          |594     |
|RX    |pharmacy_npi  |13        |9            |11      |
|RX    |prescriber_npi|13        |10           |11      |
+------+--------------+----------+-------------+--------+



# HCP link: claims NPI → dmn_cust → OPAL → L2_L3 account
##  build the NPI → J&J ID lookup from dmn_cust

- Takes npi_id → jnj_id from dmn_cust, with customer type and primary specialty for information.
- Business rule from the data-sources file: if one NPI has several jnj_ids, keep the one with the latest efftv_dt.
- Prints how many NPIs had more than one J&J ID.
- Check only: cust_stat_nm and end_dt value counts, which should be only Active / 2099-12-31 because of the upstream filter.

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dmn.groupBy("cust_stat_nm").count().show(truncate=False)
dmn.groupBy(F.to_date("end_dt").alias("end_dt")).count().orderBy(F.desc("count")).show(5)

d = (dmn.select(F.trim(F.col("npi_id").cast("string")).alias("npi"), F.col("jnj_id").cast("string").alias("jnj_id"),
                F.to_date("efftv_dt").alias("efftv_dt"), "cust_super_typ_nm", "cust_typ_nm", "prim_spclt_nm")
        .filter("npi IS NOT NULL AND npi != '' AND jnj_id IS NOT NULL"))
npi_jnj = (d.withColumn("n_jnj_ids", F.size(F.collect_set("jnj_id").over(Window.partitionBy("npi"))))
            .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
            .filter("rk = 1").drop("rk").cache())
print("NPIs in dmn_cust:", npi_jnj.count(), "| NPIs with more than one jnj_id:", npi_jnj.filter("n_jnj_ids > 1").count())

+------------+--------+
|cust_stat_nm|count   |
+------------+--------+
|Active      |22310923|
+------------+--------+

+----------+--------+
|    end_dt|   count|
+----------+--------+
|2099-12-31|22310923|
+----------+--------+

NPIs in dmn_cust: 6863335 | NPIs with more than one jnj_id: 40099


# Prepare OPAL

OPAL columns are read as text:
- 0 = HCP J&J ID;
- 1..5 = parent ID at each level;
- L2_L3 = the account (L2 if present, else L3, as delivered).

OPAL facts:
- rows and distinct HCPs;
- HCPs on more than one row;
- rows with L2 vs only L3 vs no account;
- whether L2_L3 really equals "L2 if present, else L3".

hcp_accounts: each HCP with its distinct L2_L3 accounts, and how many accounts each HCP has.

In [9]:
def txt(c):
    col = F.trim(F.col(f"`{c}`").cast("string"))
    return F.when(col != "", col).alias(c)

o = opal.select(*[txt(c) for c in ["0", "1", "2", "3", "4", "5", "L2_L3"]], F.col("child_lvl").cast("int").alias("child_lvl"))
o.groupBy("child_lvl").count().show()
o.agg(F.count("*").alias("rows"), F.countDistinct("0").alias("distinct_hcps"),
      F.sum(F.col("2").isNotNull().cast("int")).alias("rows_with_L2"),
      F.sum((F.col("2").isNull() & F.col("3").isNotNull()).cast("int")).alias("rows_L3_only"),
      F.sum(F.col("L2_L3").isNull().cast("int")).alias("rows_without_L2_L3"),
      F.sum((F.coalesce("2", "3") != F.col("L2_L3")).cast("int")).alias("L2_L3_not_equal_L2_else_L3")).show()

hcp_accounts = (o.filter(F.col("0").isNotNull() & F.col("L2_L3").isNotNull())
                 .select(F.col("0").alias("jnj_id"), F.col("L2_L3").alias("account_id")).distinct()
                 .withColumn("n_accounts", F.count("*").over(Window.partitionBy("jnj_id"))).cache())
hcp_accounts.select("jnj_id", "n_accounts").distinct().groupBy("n_accounts").count().orderBy("n_accounts").show()

+---------+-------+
|child_lvl|  count|
+---------+-------+
|        0|4146177|
+---------+-------+

+-------+-------------+------------+------------+------------------+--------------------------+
|   rows|distinct_hcps|rows_with_L2|rows_L3_only|rows_without_L2_L3|L2_L3_not_equal_L2_else_L3|
+-------+-------------+------------+------------+------------------+--------------------------+
|4146177|      4146177|     2357156|     1746272|             42749|                         0|
+-------+-------------+------------+------------+------------------+--------------------------+

+----------+-------+
|n_accounts|  count|
+----------+-------+
|         1|4103428|
+----------+-------+



# HCP-link coverage, per NPI role

For each NPI role on the claims, counts how many distinct NPIs:

- exist on the claims;
- are found in dmn_cust (get a J&J ID);
- have that J&J ID in OPAL column 0;
- reach at least one L2_L3 account.

Also shows the dmn_cust customer type of the matched NPIs (doctor vs organization).

Look at: the % reaching an account for rendering_npi, the doctor on medical claims (confirmed by Ravi), and 
prescriber_npi (pharmacy, RX-only patients). This is the key number for the doctor-to-account link.

- The other roles are kept for information only.
- Medical claims with an empty rendering_npi have no doctor; section 2 shows how many there are (empty rendering_npi).

In [10]:
opal_hcps = o.filter(F.col("0").isNotNull()).select(F.col("0").alias("jnj_id")).distinct().withColumn("in_opal", F.lit(1))
acct_hcps = hcp_accounts.select("jnj_id").distinct().withColumn("has_account", F.lit(1))

npi_cov = (claim_npis.select("source", "npi_role", "npi").distinct()
           .join(npi_jnj.select("npi", "jnj_id", "cust_typ_nm"), "npi", "left")
           .join(opal_hcps, "jnj_id", "left").join(acct_hcps, "jnj_id", "left"))
(npi_cov.groupBy("source", "npi_role")
        .agg(F.count("*").alias("npis"),
             F.sum(F.col("jnj_id").isNotNull().cast("int")).alias("in_dmn_cust"),
             F.sum(F.coalesce("in_opal", F.lit(0))).alias("in_opal"),
             F.sum(F.coalesce("has_account", F.lit(0))).alias("reach_L2_L3"))
        .withColumn("pct_reach_L2_L3", F.round(100 * F.col("reach_L2_L3") / F.col("npis"), 1))
        .orderBy("source", "npi_role").show(truncate=False))
(npi_cov.groupBy("npi_role", F.coalesce("cust_typ_nm", F.lit("NOT IN DMN_CUST")).alias("dmn_cust_type"))
        .count().orderBy("npi_role", F.desc("count")).show(40, False))

+------+--------------+----+-----------+-------+-----------+---------------+
|source|npi_role      |npis|in_dmn_cust|in_opal|reach_L2_L3|pct_reach_L2_L3|
+------+--------------+----+-----------+-------+-----------+---------------+
|MX    |billing_npi   |208 |157        |0      |0          |0.0            |
|MX    |referring_npi |74  |73         |72     |68         |91.9           |
|MX    |rendering_npi |505 |500        |496    |477        |94.5           |
|RX    |pharmacy_npi  |9   |8          |0      |0          |0.0            |
|RX    |prescriber_npi|10  |10         |10     |10         |100.0          |
+------+--------------+----+-----------+-------+-----------+---------------+

+--------------+------------------------+-----+
|npi_role      |dmn_cust_type           |count|
+--------------+------------------------+-----+
|billing_npi   |Group Practice          |77   |
|billing_npi   |Hospital                |62   |
|billing_npi   |NOT IN DMN_CUST         |51   |
|billing_npi   |Cl

# Link DDD jnj_id → OPAL → L2_L3 account

## Find where the DDD buyers sit in OPAL

Lists every ID in OPAL with the column(s) it appears in (0 to 5).

Looks up each DDD jnj_id (Rybrevant rows only) and groups the DDD units by what was found:

- L1 → 1 account: the ID is an outlet in OPAL column 1 and all its rows point to one L2_L3;
- L1 → several accounts: the outlet points to more than one L2_L3 (needs a rule, see section 10);
- is L2 / is L3: the ID is itself an account in column 2 or 3;
- other OPAL level / not in OPAL.

Look at: the % of units that can be placed on exactly one account.

In [11]:
opal_ids = None
for lvl in ["0", "1", "2", "3", "4", "5"]:
    part = o.filter(F.col(lvl).isNotNull()).select(F.col(lvl).alias("jnj_id"), F.lit(lvl).alias("col"))
    opal_ids = part if opal_ids is None else opal_ids.unionByName(part)
opal_ids = (opal_ids.distinct().groupBy("jnj_id")
            .agg(F.concat_ws("+", F.sort_array(F.collect_set("col"))).alias("opal_columns")))

outlet_acct = (o.filter(F.col("1").isNotNull() & F.col("L2_L3").isNotNull())
                .groupBy(F.col("1").alias("jnj_id"))
                .agg(F.countDistinct("L2_L3").alias("n_accounts"), F.min("L2_L3").alias("only_account")))

ddd_ids = (ddd_r.groupBy(F.col("jnj_id").cast("string").alias("jnj_id"))
                .agg(F.sum("equiv_uns").alias("equiv_uns"))
                .join(opal_ids, "jnj_id", "left").join(outlet_acct, "jnj_id", "left"))
ddd_ids = ddd_ids.withColumn("link",
    F.when(F.col("n_accounts") == 1, "L1 -> 1 account")
     .when(F.col("n_accounts") > 1, "L1 -> several accounts")
     .when(F.col("opal_columns").contains("2"), "is L2")
     .when(F.col("opal_columns").contains("3"), "is L3")
     .when(F.col("opal_columns").isNotNull(), "other OPAL column: " )
     .otherwise("not in OPAL"))
ddd_ids = ddd_ids.withColumn("link", F.when(F.col("link") == "other OPAL column: ",
                                            F.concat(F.col("link"), F.col("opal_columns"))).otherwise(F.col("link"))).cache()
(ddd_ids.groupBy("link").agg(F.count("*").alias("ddd_ids"), F.round(F.sum("equiv_uns"), 2).alias("equiv_uns"))
        .withColumn("pct_units", F.round(100 * F.col("equiv_uns") / F.sum("equiv_uns").over(Window.partitionBy()), 1))
        .orderBy(F.desc("equiv_uns")).show(truncate=False))

+--------------------+-------+---------+---------+
|link                |ddd_ids|equiv_uns|pct_units|
+--------------------+-------+---------+---------+
|L1 -> 1 account     |608    |1550.46  |56.1     |
|not in OPAL         |594    |1202.58  |43.5     |
|other OPAL column: 1|4      |10.18    |0.4      |
+--------------------+-------+---------+---------+



# billing / pharmacy NPI → dmn_cust → DDD

Check whether claim organizations are DDD buyers. If so, claim could link to sales directly, without going through HCP. This is a cross check.

Takes the organization NPIs on the claims: billing_npi, facility_npi (medical) and pharmacy_npi (pharmacy).

- Looks them up in dmn_cust to get a J&J ID, then checks:
- is that J&J ID a Rybrevant DDD buyer?
- in which OPAL column does it appear?
- Look at: if many billing NPIs are DDD buyers, claims can be linked to sales directly. That would be a second route, a cross-check of the doctor route.

In [12]:
ddd_buyers = ddd_r.select(F.col("jnj_id").cast("string").alias("jnj_id")).distinct().withColumn("is_ddd_buyer", F.lit(True))
org = (claim_npis.filter(F.col("npi_role").isin("billing_npi", "facility_npi", "pharmacy_npi"))
       .groupBy("npi_role", "npi").agg(F.countDistinct("patient_id").alias("patients"))
       .join(npi_jnj.select("npi", "jnj_id", "cust_typ_nm"), "npi", "left")
       .join(ddd_buyers, "jnj_id", "left").join(opal_ids, "jnj_id", "left")
       .fillna({"is_ddd_buyer": False}))
(org.groupBy("npi_role", F.col("jnj_id").isNotNull().alias("in_dmn_cust"), "is_ddd_buyer", "opal_columns")
    .agg(F.count("*").alias("org_npis"), F.sum("patients").alias("patients"))
    .orderBy("npi_role", F.desc("org_npis")).show(40, False))

+------------+-----------+------------+------------+--------+--------+
|npi_role    |in_dmn_cust|is_ddd_buyer|opal_columns|org_npis|patients|
+------------+-----------+------------+------------+--------+--------+
|billing_npi |true       |false       |2           |79      |252     |
|billing_npi |true       |false       |3           |61      |170     |
|billing_npi |false      |false       |null        |51      |124     |
|billing_npi |true       |false       |null        |14      |52      |
|billing_npi |true       |false       |5           |3       |3       |
|pharmacy_npi|true       |false       |2           |5       |6       |
|pharmacy_npi|true       |false       |null        |3       |4       |
|pharmacy_npi|false      |false       |null        |1       |1       |
+------------+-----------+------------+------------+--------+--------+



# list every diagnosis code on the Rybrevant claims, with no classification
- Medical claims: diagnosis_codes is a |-separated list, so it is split into single codes.
- Pharmacy claims: diagnosis_code, for RX-only patients.
- For each code: number of claims and patients, by source (M / RX) and product.
- Also: how many claims and patients have no diagnosis code.
- Nothing is labelled Lung or H&N here. This table need to be marked which codes are Lung, H&N, or neither. His list then drives the labelling.
- To export it, see the save cell at the end (dx_inventory).

In [13]:

mx_dx = mx_r.select(F.lit("MX").alias("source"), F.col("mx_claim_id").alias("claim_id"),
                    F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "product",
                    F.explode_outer(F.split(F.coalesce(F.col("diagnosis_codes"), F.lit("")), r"\|")).alias("dx_code"))
rx_dx = rx_r.select(F.lit("RX").alias("source"), "claim_id",
                    F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "product",
                    F.col("diagnosis_code").alias("dx_code"))
dx = mx_dx.unionByName(rx_dx).withColumn("dx_code", F.upper(F.trim("dx_code")))

no_dx = (dx.groupBy("source", "claim_id", "patient_id")
           .agg(F.max(F.when(F.col("dx_code") != "", 1).otherwise(0)).alias("has_dx")))
(no_dx.groupBy("source").agg(F.count("*").alias("claims"),
                             F.sum((F.col("has_dx") == 0).cast("int")).alias("claims_without_dx")).show())

dx_inventory = (dx.filter(F.col("dx_code").isNotNull() & (F.col("dx_code") != ""))
                  .groupBy("dx_code", F.substring("dx_code", 1, 3).alias("dx_3char"), "source", "product")
                  .agg(F.countDistinct("claim_id").alias("claims"), F.countDistinct("patient_id").alias("patients"))
                  .orderBy(F.desc("patients")).cache())
print("distinct diagnosis codes:", dx_inventory.select("dx_code").distinct().count())
dx_inventory.show(100, False)

+------+------+-----------------+
|source|claims|claims_without_dx|
+------+------+-----------------+
|    MX|   983|                1|
|    RX|    13|                7|
+------+------+-----------------+

distinct diagnosis codes: 468
+-------+--------+------+----------------+------+--------+
|dx_code|dx_3char|source|product         |claims|patients|
+-------+--------+------+----------------+------+--------+
|Z5111  |Z51     |MX    |RYBREVANT FASPRO|240   |162     |
|Z5111  |Z51     |MX    |RYBREVANT       |207   |147     |
|Z5112  |Z51     |MX    |RYBREVANT FASPRO|248   |141     |
|Z5112  |Z51     |MX    |RYBREVANT       |138   |103     |
|C7951  |C79     |MX    |RYBREVANT FASPRO|137   |95      |
|C3490  |C34     |MX    |RYBREVANT FASPRO|142   |89      |
|C3490  |C34     |MX    |RYBREVANT       |90    |70      |
|C7951  |C79     |MX    |RYBREVANT       |83    |66      |
|Z79899 |Z79     |MX    |RYBREVANT FASPRO|82    |58      |
|C3411  |C34     |MX    |RYBREVANT FASPRO|104   |56      

## Put DDD units and claim doctors side by side per L2_L3 account

- What is baught (DDD units of Ryb)
- Who treats there (how many HCPs and patients from the claims)
- WHat claims show it infused (number of infused and mg)


Sales side: Rybrevant and Rybrevant Faspro DDD units per account.

- Only DDD IDs with one clear account are used: "L1 → 1 account", "is L2", "is L3".
- Ambiguous IDs are left out and counted in section 5.

Doctor side: for each account, the number of distinct doctors that reach it, and their patients:

- medical claims: rendering_npi (confirmed by Ravi);
- pharmacy claims: prescriber_npi, only for RX-only patients (patients already in MX were removed in section 3), shown in its own column.
- A doctor with several accounts is counted fully at each one (no split). How to split them is an open question.

Then: how many accounts have sales and doctors, sales only, or doctors only, with their share of units.

Look at: the share of units in accounts that also have claim doctors. That is how much of the sales can later be split by the doctors' Lung / H&N mix.

In [14]:
ddd_acct_map = (ddd_ids.filter(F.col("link").isin("L1 -> 1 account", "is L2", "is L3"))
                       .select("jnj_id", F.when(F.col("link") == "L1 -> 1 account", F.col("only_account"))
                                          .otherwise(F.col("jnj_id")).alias("account_id")))
acct_sales = (ddd_r.withColumn("jnj_id", F.col("jnj_id").cast("string")).join(ddd_acct_map, "jnj_id")
              .groupBy("account_id")
              .pivot("product").agg(F.round(F.sum("equiv_uns"), 3)).fillna(0))
unit_cols = [c for c in acct_sales.columns if c != "account_id"]
acct_sales = acct_sales.withColumn("total_ryb_units", sum(F.col(f"`{c}`") for c in unit_cols))

hcp_roles = ["rendering_npi", "prescriber_npi"]   # doctor = rendering NPI on medical claims (Ravi); prescriber on pharmacy claims
acct_docs = (claim_npis.filter(F.col("npi_role").isin(hcp_roles))
             .join(npi_jnj.select("npi", "jnj_id"), "npi").join(hcp_accounts, "jnj_id")
             .groupBy("account_id").pivot("npi_role", hcp_roles)
             .agg(F.countDistinct("npi").alias("npis"), F.countDistinct("patient_id").alias("patients"))
             .fillna(0))

accounts = acct_sales.join(acct_docs, "account_id", "full").fillna(0)
doc_cols = [c for c in accounts.columns if c.endswith("_npis")]
accounts = accounts.withColumn("has_sales", F.col("total_ryb_units") > 0) \
                   .withColumn("has_doctors", sum(F.col(c) for c in doc_cols) > 0)
(accounts.groupBy("has_sales", "has_doctors")
         .agg(F.count("*").alias("accounts"), F.round(F.sum("total_ryb_units"), 2).alias("units"))
         .withColumn("pct_units", F.round(100 * F.col("units") / F.sum("units").over(Window.partitionBy()), 1))
         .orderBy(F.desc("units")).show())
accounts.orderBy(F.desc("total_ryb_units")).show(25, False)

+---------+-----------+--------+-------+---------+
|has_sales|has_doctors|accounts|  units|pct_units|
+---------+-----------+--------+-------+---------+
|     true|      false|     464|1095.51|     70.7|
|     true|       true|     125| 454.97|     29.3|
|    false|      false|       3|    0.0|      0.0|
|    false|       true|     246|    0.0|      0.0|
+---------+-----------+--------+-------+---------+

+----------+---------+----------------+------------------+------------------+----------------------+-------------------+-----------------------+---------+-----------+
|account_id|RYBREVANT|RYBREVANT FASPRO|total_ryb_units   |rendering_npi_npis|rendering_npi_patients|prescriber_npi_npis|prescriber_npi_patients|has_sales|has_doctors|
+----------+---------+----------------+------------------+------------------+----------------------+-------------------+-----------------------+---------+-----------+
|199518415 |3.765    |24.0            |27.765            |0                 |0            

# Diagnosis.........

In [4]:
from pyspark.sql import functions as F
dup = (mx.groupBy("dv1", "dv2", "service_date", "ndc")
         .agg(F.count("*").alias("lines"),
              F.countDistinct("data_source").alias("n_sources"),
              F.collect_set("data_source").alias("sources"),
              F.collect_set("modifiers").alias("modifiers"),
              F.sum("quantity").alias("total_qty")))
dup.groupBy("lines", "n_sources").count().orderBy("lines", "n_sources").show()      # how common is each case
dup.filter("n_sources > 1").show(20, False)                                          # professional + facility pairs
dup.filter("lines > 1 AND n_sources = 1").show(20, False)                            # same source, e.g. JW waste lines



+-----+---------+-----+
|lines|n_sources|count|
+-----+---------+-----+
|    1|        1|  812|
|    2|        1|   84|
|    3|        1|    1|
+-----+---------+-----+

+---+---+------------+---+-----+---------+-------+---------+---------+
|dv1|dv2|service_date|ndc|lines|n_sources|sources|modifiers|total_qty|
+---+---+------------+---+-----+---------+-------+---------+---------+
+---+---+------------+---+-----+---------+-------+---------+---------+

+--------------------------------------------+--------------------------------------------+-------------------+-----------+-----+---------+---------------+---------------------+---------+
|dv1                                         |dv2                                         |service_date       |ndc        |lines|n_sources|sources        |modifiers            |total_qty|
+--------------------------------------------+--------------------------------------------+-------------------+-----------+-----+---------+---------------+---------------

In [5]:
ex = mx.filter(
    (F.col("dv1").startswith("0/Qp5ClX") & (F.to_date("service_date") == "2026-08-11") & (F.col("ndc") == "57894052201")) |
    (F.col("dv1").startswith("05Q7uvRB") & (F.to_date("service_date") == "2026-08-17") & (F.col("ndc") == "57894050101")))
(ex.select(F.substring("dv1", 1, 8).alias("patient"), F.to_date("service_date").alias("service_date"), "ndc",
           "data_source", "modifiers", "quantity", "unit_type", "procedure_code", "rendering_npi", "mx_claim_id")
   .orderBy("patient", "modifiers").show(truncate=False))


+--------+------------+-----------+-------------+----------+--------+---------+--------------+-------------+-----------------+
|patient |service_date|ndc        |data_source  |modifiers |quantity|unit_type|procedure_code|rendering_npi|mx_claim_id      |
+--------+------------+-----------+-------------+----------+--------+---------+--------------+-------------+-----------------+
|0/Qp5ClX|2026-08-11  |57894052201|INSTITUTIONAL||JW|TB|PO||32.0    |10 MG    |J9062         |1730660408   |I36ppq77dszC00009|
|0/Qp5ClX|2026-08-11  |57894052201|INSTITUTIONAL||TB|PO|   |672.0   |10 MG    |J9062         |1730660408   |I36ppq77dszC00010|
|05Q7uvRB|2026-08-17  |57894050101|PROFESSIONAL ||76|JZ|   |437.0   |2 MG     |J9061         |1144240334   |P06yhcv4eg8C00017|
|05Q7uvRB|2026-08-17  |57894050101|PROFESSIONAL ||JZ|      |438.0   |2 MG     |J9061         |1144240334   |P06yhcv4eg8C00014|
+--------+------------+-----------+-------------+----------+--------+---------+--------------+-------------+---

In [11]:
#Q3:
# Patients in both MX and RX: how far is each RX fill from the patient's nearest MX claim?
mx_days = mx_r.select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"),
                      F.to_date("service_date").alias("mx_day"),
                      F.col("rendering_npi").alias("mx_npi")).distinct()
rx_x = rx_all.select("patient_id", "claim_id", F.to_date("fill_date").alias("rx_day"), "prescriber_npi")

both = rx_x.join(mx_days.select("patient_id").distinct(), "patient_id", "left_semi")
gaps = (both.join(mx_days, "patient_id")
            .groupBy("patient_id", "claim_id", "rx_day", "prescriber_npi")
            .agg(F.min(F.abs(F.datediff("rx_day", "mx_day"))).alias("days_to_nearest_mx"),
                 F.max((F.col("prescriber_npi") == F.col("mx_npi")).cast("int")).alias("same_doctor_as_mx")))

# 1. Overall picture
print("RX patients:", rx_all.select("patient_id").distinct().count(),
      "| also in MX:", both.select("patient_id").distinct().count(),
      "| RX rows of those patients:", both.count())

# 2. Distance of each RX row from the nearest MX claim
(gaps.withColumn("gap_bucket", F.when(F.col("days_to_nearest_mx") == 0, "0 same day")
                                .when(F.col("days_to_nearest_mx") <= 7, "1-7 days")
                                .when(F.col("days_to_nearest_mx") <= 30, "8-30 days")
                                .when(F.col("days_to_nearest_mx") <= 90, "31-90 days")
                                .otherwise("> 90 days"))
     .groupBy("gap_bucket")
     .agg(F.count("*").alias("rx_rows"), F.countDistinct("patient_id").alias("patients"),
          F.sum("same_doctor_as_mx").alias("rows_same_doctor_as_mx"))
     .orderBy("gap_bucket").show(truncate=False))

# 3. One example patient: MX and RX dates side by side
ex = both.select("patient_id").distinct().limit(1)
(mx_days.join(ex, "patient_id").select("patient_id", F.lit("MX").alias("src"), F.col("mx_day").alias("day"), F.col("mx_npi").alias("npi"))
 .unionByName(rx_x.join(ex, "patient_id").select("patient_id", F.lit("RX").alias("src"), F.col("rx_day").alias("day"), F.col("prescriber_npi").alias("npi")))
 .withColumn("patient_id", F.substring("patient_id", 1, 8)).orderBy("day").show(50, False))




RX patients: 11 | also in MX: 1 | RX rows of those patients: 1
+----------+-------+--------+----------------------+
|gap_bucket|rx_rows|patients|rows_same_doctor_as_mx|
+----------+-------+--------+----------------------+
|31-90 days|1      |1       |1                     |
+----------+-------+--------+----------------------+

+----------+---+----------+----------+
|patient_id|src|day       |npi       |
+----------+---+----------+----------+
|g/vUn6rZ  |RX |2026-07-13|1851346928|
|g/vUn6rZ  |MX |2026-08-13|1851346928|
|g/vUn6rZ  |MX |2026-08-27|1851346928|
+----------+---+----------+----------+

